# 03 · Baselines y LightGBM (sección 5)

| Modelo | Tipo | Implementación |
|---|---|---|
| Naive | repite la última hora observada | propio (sin parámetros) |
| Seasonal naive 24 h / 168 h | repite ayer / la semana pasada | propio, validado contra StatsForecast en el notebook 02 |
| MSTL | estacionalidad diaria + semanal, tendencia AutoETS, uno por cliente | StatsForecast |
| LightGBM global | rezagos 1–24 y 168 + hora, día, mes y feriado, recursivo | mlforecast + LightGBM |

Todos pasan por el mismo `backtest()` (92 orígenes diarios, prueba oct–dic 2014).

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import pandas as pd
from src import data as D, metrics as M, backtest as B
from src.models import baselines as BL

panel = D.build_panel()
scale = M.mase_scale(panel)
filas = []          # tabla de resultados de este notebook

## 1. Naive y seasonal naive

No tienen parámetros: el "entrenamiento" es nulo y el pronóstico solo usa la historia hasta las 23:00 del día anterior.

In [ ]:
for name, fn, L in [("Naive", BL.naive_fn, 1),
                    ("SeasonalNaive24", BL.seasonal_naive_fn(24), 24),
                    ("SeasonalNaive168", BL.seasonal_naive_fn(168), 168)]:
    fc = B.backtest(fn, panel, name=name)
    row, _ = B.finish(name, fc, panel, scale, familia="Baseline", L=L, estrategia="directa (copia)",
                      covariables=False, params_total=0, params_entrenables=0, train_s=0.0)
    filas.append(row)
pd.DataFrame(filas)[["MAE", "RMSE", "WAPE", "MASE", "MASE_navidad"]]

## 2. MSTL

MSTL descompone cada serie en tendencia + estacionalidad diaria (24) + semanal (168) y pronostica la tendencia con AutoETS. Es un modelo **local** (uno por cliente). Ajustarlo con los 2.5 años completos en cada uno de los 92 orígenes tarda más de 1 minuto por cliente, así que en cada origen se ajusta sobre las **últimas 8 semanas** (8 ciclos semanales completos). Solo usa datos anteriores al origen.

In [ ]:
fc = B.backtest(BL.mstl_fn(weeks=8), panel, name="MSTL")
row, ev_mstl = B.finish("MSTL", fc, panel, scale, familia="Baseline (estadístico)", L=8 * 168,
                        estrategia="directa (modelo de estado)", covariables=False,
                        params_total=None, params_entrenables=None,
                        train_s=0.0, nota="se reajusta por cliente en cada origen; el tiempo está en inferencia")
filas.append(row)
row[["MAE", "WAPE", "MASE", "MASE_navidad", "inference_s"]]

## 3. LightGBM global

- **Global:** un solo modelo para los 150 clientes. El objetivo se normaliza por cliente (z-score con estadísticos del bloque de entrenamiento) para que los clientes grandes no dominen.
- **Variables:** rezagos 1–24 y 168; hora, día de la semana y mes (de la fecha) y feriado de Portugal como exógena conocida a futuro.
- **Estrategia multi-paso recursiva:** predice h = 1, lo usa como rezago para h = 2, y así hasta h = 24.
- Se entrena **una vez**; en cada origen `predict(new_df=historia)` actualiza los rezagos sin reentrenar.

### Selección de configuración (solo con validación)

Se entrena con datos hasta 2014-06-30 y se evalúa con el mismo backtest diario sobre julio–septiembre.

In [ ]:
configs = {
    "lgb_a": dict(num_leaves=63, n_estimators=400, learning_rate=0.05),
    "lgb_b": dict(num_leaves=127, n_estimators=600, learning_rate=0.05),
    "lgb_c": dict(num_leaves=255, n_estimators=600, learning_rate=0.05),
}
train_panel = panel.loc[panel["ds"] <= D.TRAIN_END]
val_rows = []
for cname, params in configs.items():
    fn, info = BL.fit_lightgbm(train_panel, params=params)
    fc_val = B.backtest(fn, panel, name=cname, start=D.VAL_START, end=D.VAL_END, verbose=False)
    r = M.evaluate(fc_val, panel, scale)["resumen"]
    val_rows.append({"config": cname, **params, "fit_s": info["fit_s"], "val_MAE": r["MAE"], "val_MASE": r["MASE"]})
val_tab = pd.DataFrame(val_rows).set_index("config")
val_tab

### Modelo final

Con la mejor configuración se reentrena una vez con todos los datos hasta 2014-09-30 (entrenamiento + validación) y se evalúa en prueba una sola vez.

In [ ]:
best = val_tab["val_MASE"].idxmin()
print("Mejor configuración en validación:", best, configs[best])
fn, info = BL.fit_lightgbm(panel.loc[panel["ds"] <= D.VAL_END], params=configs[best])
fc = B.backtest(fn, panel, name="LightGBM")
row, ev_lgb = B.finish("LightGBM", fc, panel, scale, familia="Árboles", L=168,
                       estrategia="recursiva", covariables=True,
                       params_total=info["params_total"], params_entrenables=info["params_total"],
                       train_s=info["fit_s"], arboles=info["arboles"], config=configs[best])
filas.append(row)
row[["MAE", "WAPE", "MASE", "MASE_navidad", "train_s", "inference_s"]]

## 4. Resumen

In [ ]:
tabla = pd.DataFrame(filas).set_index("modelo")
cols = ["familia", "L", "estrategia", "MAE", "RMSE", "WAPE", "MAPE", "MASE", "MASE_navidad", "train_s", "inference_s"]
tabla[cols].sort_values("MASE")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 3.5))
for name in tabla.index:
    ev = M.evaluate(pd.read_parquet(B.FORECASTS_DIR / f"{name}.parquet"), panel, scale)
    ax.plot(ev["por_h"].index, ev["por_h"]["MAE"], marker="o", ms=3, label=name)
ax.set_xlabel("Paso del horizonte h"); ax.set_ylabel("MAE (kWh)")
ax.set_title("MAE por paso del horizonte, baselines y LightGBM"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()